# Grounding gate — stop unsupported claims before your product ships them

An LLM answers from your retrieved context, and some of what it says may be made
up — reading exactly as fluently as the real parts. [TokenPath](https://tokenpath.ai)
measures, from model attention, which exact source span each part of the answer
came from. A claim whose attribution traces back to nothing is very likely
fabricated — and you can gate on that *before* the answer reaches a user.

This notebook builds a `grounding_gate()` you can drop after **any** generation
call, then wires it into **LangChain** and **LlamaIndex**.

The pattern is two stages:

1. **Attribution (TokenPath)** — for each claim in the answer, find the source
   span behind it and a confidence. Low confidence ⇒ the claim rests on nothing.
2. **Verification (optional, one narrow LLM call per claim)** — attribution tells
   you *where* a claim came from, not whether it's *true*. For claims that do
   attribute, compare claim vs. its source span to catch distortions
   (source says "18%", answer says "24%"). Because stage 1 already pinned down
   the relevant source span, this check is tiny, cheap, and far more reliable
   than fact-checking a whole answer against a whole document.

## Setup

You need a TokenPath API key — free at [platform.tokenpath.ai](https://platform.tokenpath.ai)
(10M attributed tokens on signup, no card required). Export it as `TOKENPATH_API_KEY`
before running this notebook.

In [1]:
%pip install -q requests

In [2]:
import os
import re

import requests

API_URL = "https://api.tokenpath.ai"
API_KEY = os.environ.get("TOKENPATH_API_KEY")
assert API_KEY, (
    "Set TOKENPATH_API_KEY first — grab a free key (10M tokens, no card) "
    "at https://platform.tokenpath.ai"
)


def attribute(document, question, answer, spans, **options):
    """Resolve each answer character span to the source span that produced it."""
    response = requests.post(
        f"{API_URL}/v1/attributions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "document": document,
            "question": question,
            "answer": answer,
            "spans": spans,
            **options,
        },
        timeout=120,
    )
    response.raise_for_status()
    return response.json()

### Splitting an answer into claims

TokenPath resolves the character spans *you* send. The simplest useful unit is a
sentence: split the answer on sentence boundaries and attribute each one.

In [3]:
def claim_spans(text):
    """Sentence-level [start, end) character spans over `text`."""
    boundary = re.compile(r"[.!?][\"\')\]]*(?=\s|$)|\n")
    raw, start = [], 0
    for m in boundary.finditer(text):
        raw.append((start, m.end()))
        start = m.end()
    if start < len(text):
        raw.append((start, len(text)))
    spans = []
    for s, e in raw:
        segment = text[s:e]
        if segment.strip():
            left = len(segment) - len(segment.lstrip())
            right = len(segment) - len(segment.rstrip())
            spans.append([s + left, e - right])
    return spans


demo = "Revenue grew 18%. Margin improved."
[demo[s:e] for s, e in claim_spans(demo)]

['Revenue grew 18%.', 'Margin improved.']

## The gate

One call to `POST /v1/attributions` resolves every claim at once. Each claim
comes back with the strongest source span and a confidence in `[0, 1]`
(or `source: null` if nothing in the document supports it).

In [4]:
def grounding_gate(document, question, answer, threshold=0.8):
    """Attribute every claim in `answer`; flag claims below `threshold`."""
    spans = claim_spans(answer)
    result = attribute(document, question, answer, spans)
    claims = []
    for item in result["spans"]:
        source = item["source"]
        confidence = source["confidence"] if source else 0.0
        claims.append(
            {
                "claim": item["answer"]["text"],
                "confidence": confidence,
                "grounded": confidence >= threshold,
                "source_text": source["text"] if source else None,
            }
        )
    flagged = [c for c in claims if not c["grounded"]]
    return {"passed": not flagged, "claims": claims, "flagged": flagged}


def print_report(report):
    for c in report["claims"]:
        mark = "PASS" if c["grounded"] else "FAIL"
        print(f"[{mark}] {c['confidence']:.2f}  {c['claim']}")
        if c["source_text"]:
            print(f"          source: {c['source_text']!r}")
        else:
            print("          source: (none — traced back to nothing)")
    print("\ngate:", "PASSED" if report["passed"] else "BLOCKED")

## A grounded answer passes

In [5]:
DOCUMENT = """NORTHWIND TRADERS — Q3 2025 SHAREHOLDER NOTE

Revenue grew 18% year over year to $412 million, driven primarily by the
Enterprise segment, which expanded 31%. Gross margin improved to 64.2%,
up from 61.8% a year ago. The company closed the quarter with 2,847
employees, up from 2,610 at the end of Q2.

Operating expenses rose 9% to $198 million, reflecting continued
investment in the Fabrikam integration, which remains on track to
complete in Q1 2026. The board approved a $50 million share buyback
program. No dividend was declared this quarter.

CEO Elena Vasquez said the company expects "mid-teens revenue growth"
for the full fiscal year, citing a record $1.2 billion contracted
backlog. Guidance assumes no material FX headwinds."""

QUESTION = "Summarize Northwind's Q3 results."

In [6]:
GROUNDED_ANSWER = (
    "Northwind's revenue grew 18% year over year to $412 million, led by 31% "
    "growth in the Enterprise segment. Gross margin improved to 64.2%. The "
    "board approved a $50 million share buyback, and the company guided to "
    "mid-teens revenue growth for the full year."
)

print_report(grounding_gate(DOCUMENT, QUESTION, GROUNDED_ANSWER))

[PASS] 0.99  Northwind's revenue grew 18% year over year to $412 million, led by 31% growth in the Enterprise segment.
          source: 'grew 18% year over year to $412 million, driven primarily by the\nEnterprise segment, which expanded 31%'
[PASS] 0.98  Gross margin improved to 64.2%.
          source: 'Gross margin improved to 64.2%'
[PASS] 0.99  The board approved a $50 million share buyback, and the company guided to mid-teens revenue growth for the full year.
          source: 'mid-teens revenue growth"\nfor the full fiscal year,'

gate: PASSED


## A fabricated claim gets caught

This answer invents a dividend and a Singapore office. Neither exists anywhere
in the document, so their attribution mass collapses and the gate blocks the
answer.

In [7]:
HALLUCINATED_ANSWER = (
    "Northwind's revenue grew 24% year over year to $450 million. The company "
    "declared a $0.10 dividend and announced a new office in Singapore. Gross "
    "margin improved to 64.2%."
)

report = grounding_gate(DOCUMENT, QUESTION, HALLUCINATED_ANSWER)
print_report(report)

[PASS] 0.98  Northwind's revenue grew 24% year over year to $450 million.
          source: 'grew 18% year over year to $412 million,'
[FAIL] 0.60  The company declared a $0.10 dividend and announced a new office in Singapore.
          source: 'a $50 million'
[PASS] 0.96  Gross margin improved to 64.2%.
          source: 'Gross margin improved to 64.2%'

gate: BLOCKED


## What the gate catches — and what needs stage 2

Look closely at the first claim above: *"revenue grew 24% year over year to
$450 million"* attributes **confidently** to the revenue sentence — because
that *is* the span it was generated from. Attribution localizes; it doesn't
judge. The numbers were distorted in flight, and only a comparison of claim
vs. source can catch that.

That's stage 2: for each claim, one narrow LLM call with just the claim and its
mapped source span — *"is this claim supported by this source text?"* — instead
of fact-checking the whole answer against the whole document. The cell below
runs it if you have `langchain-openai` installed and `OPENAI_API_KEY` set;
otherwise it prints the prompt so you can wire in any model you like.

In [8]:
VERIFIER_PROMPT = """You are checking one claim against the source text it was \
generated from. Reply with exactly one word — SUPPORTED, DISTORTED, or \
UNSUPPORTED — followed by a one-sentence reason.

Claim: {claim}
Source text: {source}"""

try:
    from langchain_openai import ChatOpenAI

    verifier = ChatOpenAI(model="gpt-4o-mini") if os.environ.get("OPENAI_API_KEY") else None
except ImportError:
    verifier = None

for c in report["claims"]:
    if c["source_text"] is None:
        print(f"UNSUPPORTED (no source) — {c['claim']}")
        continue
    prompt = VERIFIER_PROMPT.format(claim=c["claim"], source=c["source_text"])
    if verifier:
        print(verifier.invoke(prompt).content, "—", c["claim"])
    else:
        print(f"(verifier skipped — set OPENAI_API_KEY) — {c['claim']}")

(verifier skipped — set OPENAI_API_KEY) — Northwind's revenue grew 24% year over year to $450 million.
(verifier skipped — set OPENAI_API_KEY) — The company declared a $0.10 dividend and announced a new office in Singapore.
(verifier skipped — set OPENAI_API_KEY) — Gross margin improved to 64.2%.


## Wiring it into LangChain

The gate is one function call after `chain.invoke()` — nothing about the chain
changes. Wrap it in a `RunnableLambda` if you want the gate to be part of the
chain itself. Requires `pip install langchain-openai` and `OPENAI_API_KEY`.

In [9]:
try:
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnableLambda
    from langchain_openai import ChatOpenAI

    HAVE_LANGCHAIN = bool(os.environ.get("OPENAI_API_KEY"))
except ImportError:
    HAVE_LANGCHAIN = False

if not HAVE_LANGCHAIN:
    print("Skipped — `pip install langchain-openai` and set OPENAI_API_KEY to run this cell.")
else:
    prompt = ChatPromptTemplate.from_template(
        "Answer using only this document:\n\n{document}\n\nQuestion: {question}"
    )
    chain = prompt | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()

    def generate_with_gate(inputs):
        answer = chain.invoke(inputs)
        gate = grounding_gate(inputs["document"], inputs["question"], answer)
        if not gate["passed"]:
            # your policy here: re-retrieve, regenerate, or escalate to a human
            return {"answer": None, "blocked_claims": gate["flagged"]}
        return {"answer": answer, "claims": gate["claims"]}

    gated_chain = RunnableLambda(generate_with_gate)
    result = gated_chain.invoke({"document": DOCUMENT, "question": QUESTION})
    print(result["answer"])

Skipped — `pip install langchain-openai` and set OPENAI_API_KEY to run this cell.


## Wiring it into LlamaIndex

Same idea. One important detail: attribute against the **retrieved context the
model actually saw** (`response.source_nodes`), not your whole corpus. Requires
`pip install llama-index` and `OPENAI_API_KEY`.

In [10]:
try:
    from llama_index.core import Document, VectorStoreIndex

    HAVE_LLAMAINDEX = bool(os.environ.get("OPENAI_API_KEY"))
except ImportError:
    HAVE_LLAMAINDEX = False

if not HAVE_LLAMAINDEX:
    print("Skipped — `pip install llama-index` and set OPENAI_API_KEY to run this cell.")
else:
    index = VectorStoreIndex.from_documents([Document(text=DOCUMENT)])
    engine = index.as_query_engine()
    response = engine.query(QUESTION)

    retrieved_context = "\n\n".join(
        node.node.get_content() for node in response.source_nodes
    )
    print_report(grounding_gate(retrieved_context, QUESTION, str(response)))

Skipped — `pip install llama-index` and set OPENAI_API_KEY to run this cell.


## Tuning notes

- **Threshold.** `confidence` is normalized to `[0, 1]` and comparable across
  claims. In our runs, claims that trace to real source text score ≳ 0.95 and
  fabricated ones fall well below 0.8 — start there and tune on your traffic.
- **Below threshold, choose a policy**: re-retrieve with a different query,
  regenerate with the flagged claim quoted back to the model, drop the claim, or
  escalate. The gate gives you the exact claim *and* its nearest source text, so
  every one of those is mechanical.
- **Long documents.** `document` caps at 400k characters. Past that, attribute
  per chunk and merge.

---

## Where to go next

- **API reference & guides** — [docs.tokenpath.ai](https://docs.tokenpath.ai)
- **Bugs / broken examples** — [open an issue](https://github.com/tokenpath/cookbook/issues)
- **"How do I…?"** — [start a discussion](https://github.com/tokenpath/cookbook/discussions)
- **Email** — support@tokenpath.ai